In [ ]:
from selenium import webdriver as wb
from bs4 import BeautifulSoup as bs
import time
from selenium.webdriver.common.by import By

In [ ]:
driver = wb.Chrome()
driver.get("https://www.bustago.or.kr/newweb/kr/terminalInfo/terminallist.do")

In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup as bs
import pandas as pd

# 웹 드라이버 설정
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

# 실제 URL로 변경해야 합니다.
base_url = "https://www.bustago.or.kr/newweb/kr/terminalInfo/terminallist.do"  # 실제 URL로 변경하세요.

# 지역 목록
regions = [
    "서울", "부산", "대구", "인천", "광주", "대전", "울산",
    "세종", "경기도", "강원도", "충청북도", "충청남도",
    "전라북도", "전라남도", "경상북도", "경상남도"
]

# 결과를 저장할 리스트 생성
data = []

def extract_table_data(soup):
    """테이블에서 데이터를 추출하여 리스트에 추가"""
    rows = soup.select("table.board_list tbody tr")
    for row in rows:
        cols = row.find_all("td")
        if len(cols) >= 2:  # 열이 두 개 이상일 때만 추출
            terminal_name = cols[0].get_text(strip=True)
            address = cols[1].get_text(strip=True)
            data.append({
                "장소": terminal_name,
                "도로명 주소": address
            })

for region in regions:
    try:
        # 페이지 로드
        driver.get(base_url)
        time.sleep(2)  # 페이지 로딩 대기

        # 'sbox' 영역의 'areaCategory' 목록 클릭
        driver.find_element(By.CSS_SELECTOR, f"div.sbox ul#areaCategory li a[title='{region}']").click()
        time.sleep(2)  # 페이지 로딩 대기

        # 마지막 페이지로 이동
        while True:
            try:
                last_button = driver.find_element(By.CSS_SELECTOR, "a.last")
                last_button.click()
                time.sleep(2)  # 페이지 로딩 대기
                break
            except:
                break

        # 마지막 페이지에서 데이터 추출
        while True:
            soup = bs(driver.page_source, "lxml")
            extract_table_data(soup)

            # '이전으로 이동' 버튼 클릭
            try:
                prev_button = driver.find_element(By.CSS_SELECTOR, "a.prev")
                if "disabled" in prev_button.get_attribute("class"):
                    break
                prev_button.click()
                time.sleep(2)  # 페이지 로딩 대기
            except:
                break

    except Exception as e:
        print(f"오류 발생 ({region}): {e}")

# 드라이버 종료
driver.quit()

# 데이터프레임으로 변환
df = pd.DataFrame(data)

# 데이터프레임을 CSV 파일로 저장
df.to_csv("terminals_data1.csv", index=False, encoding='utf-8-sig')

print("데이터가 'terminals_data1.csv' 파일에 저장되었습니다.")


In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup as bs
import pandas as pd

# 웹 드라이버 설정
driver = webdriver.Chrome()

# 실제 URL
base_url = "https://www.bustago.or.kr/newweb/kr/terminalInfo/terminallist.do"

# 지역 목록
regions = [
    "서울", "부산", "대구", "인천", "광주", "대전", "울산",
    "세종", "경기도", "강원도", "충청북도", "충청남도",
    "전라북도", "전라남도", "경상북도", "경상남도"
]

# 결과를 저장할 리스트 생성
data = []

def extract_table_data(soup):
    """테이블에서 데이터를 추출하여 리스트에 추가"""
    rows = soup.select("table.board_list tbody tr")
    for row in rows:
        cols = row.find_all("td")
        if len(cols) >= 5:  # 열이 다섯 개 이상일 때만 추출
            number = cols[0].get_text(strip=True)
            area = cols[1].get_text(strip=True)
            terminal_name = cols[2].get_text(strip=True)
            address = cols[3].get_text(strip=True)
            contact = cols[4].get_text(strip=True) if len(cols) > 4 else ""
            data.append({
                "번호": number,
                "지역": area,
                "터미널명": terminal_name,
                "주소": address,
                "연락처": contact
            })

for region in regions:
    try:
        # 페이지 로드
        driver.get(base_url)
        time.sleep(2)  # 페이지 로딩 대기

        # 'sbox' 영역의 'areaCategory' 목록 클릭
        driver.find_element(By.CSS_SELECTOR, f"div.sbox ul#areaCategory li a[title='{region}']").click()
        time.sleep(2)  # 페이지 로딩 대기

        # 마지막 페이지로 이동
        while True:
            try:
                last_button = driver.find_element(By.CSS_SELECTOR, "a.last")
                last_button.click()
                time.sleep(2)  # 페이지 로딩 대기
                soup = bs(driver.page_source, "lxml")
                extract_table_data(soup)
                # 이전 페이지로 돌아가기 전에 마지막 페이지의 데이터 크롤링
                while True:
                    try:
                        prev_button = driver.find_element(By.CSS_SELECTOR, "a.prev")
                        if "disabled" in prev_button.get_attribute("class"):
                            break
                        prev_button.click()
                        time.sleep(2)  # 페이지 로딩 대기
                        soup = bs(driver.page_source, "lxml")
                        extract_table_data(soup)
                    except:
                        break
                break
            except:
                break

    except Exception as e:
        print(f"오류 발생 ({region}): {e}")

# 드라이버 종료
driver.quit()

# 데이터프레임으로 변환
df = pd.DataFrame(data)

# 데이터프레임을 CSV 파일로 저장
df.to_csv("terminals_data1.csv", index=False, encoding='utf-8-sig')

print("데이터가 'terminals_data1.csv' 파일에 저장되었습니다.")


In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
import time

# CSV 파일 경로
input_file_path = r'C:\Users\kfq\Downloads\1ST_Final_Project\크롤링\버스타고 크롤링\시외고속버스터미널 (버스타고).csv'
output_file_path = r'C:\Users\kfq\Downloads\1ST_Final_Project\크롤링\버스타고 크롤링\시외고속버스터미널 (버스타고)_geo.csv'

# CSV 파일 읽기
try:
    df = pd.read_csv(input_file_path, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(input_file_path, encoding='cp949')

# 지오코더 초기화
geolocator = Nominatim(user_agent="socar_geocoder", timeout=10)  # timeout 값을 10초로 설정

# 지오코딩 함수 정의
def geocode_address(address, retries=5):
    for attempt in range(retries):
        try:
            location = geolocator.geocode(address)
            if location:
                return location.latitude, location.longitude
        except GeocoderTimedOut:
            print(f"Geocoding timed out on attempt {attempt + 1}. Retrying...")
            time.sleep(2)  # 2초 대기 후 재시도
        except GeocoderServiceError as e:
            print(f"Geocoding service error: {e}")
        except Exception as e:
            print(f"Unexpected error: {e}")
    return None, None  # 재시도 후에도 실패하면 None 반환

# 위도와 경도 컬럼 추가
latitudes = []
longitudes = []

for index, row in df.iterrows():
    lat, lon = None, None
    if pd.notna(row['주소']):
        lat, lon = geocode_address(row['주소'])
    
    latitudes.append(lat)
    longitudes.append(lon)

df['위도'] = latitudes
df['경도'] = longitudes

# 위도와 경도만 포함된 새로운 데이터프레임 생성
df_geocoded = df[['위도', '경도']]

# 결과를 새로운 CSV 파일로 저장 (utf-8 인코딩)
df_geocoded.to_csv(output_file_path, index=False, encoding='utf-8')

print("지오코딩이 완료되었습니다. 결과가 저장되었습니다:", output_file_path)
